# Early detection of diabetic retinopathy (grading only)
Run the cells **top to bottom**. Each section is one stage of the thesis plan.
Turn on **GPU** and **Internet** (for the ImageNet weights) in the Kaggle notebook settings, and add these datasets:
DDR, EyePACS (diabetic-retinopathy-detection), IDRiD, APTOS 2019.

In [ ]:
!pip install -q timm
BRANCH = "claude/focused-hamilton-3yd4qo"     # change to "main" after merging
!git clone -q -b $BRANCH https://github.com/kazimab1/Early-DR.git /kaggle/working/code
%cd /kaggle/working/code

## 0. Dataset locations
Edit the four paths so they match the datasets you added (look in the right-hand *Input* panel).
The prepared data and all results are written to `/kaggle/working`.

In [ ]:
import os
for name in os.listdir("/kaggle/input"):
    print("/kaggle/input/" + name)

In [ ]:
PATHS = ("paths.ddr=/kaggle/input/ddr-dataset "
         "paths.eyepacs=/kaggle/input/diabetic-retinopathy-detection "
         "paths.idrid=/kaggle/input/idrid "
         "paths.aptos=/kaggle/input/aptos2019-blindness-detection")

## 1. Data: read the datasets, freeze patient-level splits, cache 512 px images
(Run once. `splits.csv` is frozen and never recomputed.)

In [ ]:
!python -m early_dr.prepare_data configs/base.yaml --set $PATHS

## 2. Data audit: class distribution, merge composition, image statistics, duplicates

In [ ]:
!python -m early_dr.audit configs/base.yaml --set $PATHS

## 3. Does the source predict the image? (source classifier AUROC)

In [ ]:
!python -m early_dr.source_check configs/base.yaml --a DDR --b EyePACS --set $PATHS

## 4. Declare the analysis
Open `DECLARED_ANALYSIS.md`, fill in the date and commit it **before** any test set is opened (rule 2).
The training stage below only fits on train and validation. Test sets are opened by the *evaluate* stage.

## 5. Merge arms (primary task: stage 1, 0 vs 1-4)
A = DDR only, B = DDR + EyePACS source-balanced, C = version 1 merge (negative control).
Training also fits the calibration on validation and freezes it. Use `--seeds 42 43 44` for the 3-seed shortlist.

In [ ]:
for arm in ["A", "B", "C"]:
    !python -m early_dr.run_experiment configs/arms/arm_{arm}.yaml --set $PATHS task=stage1 --stage train --seeds 42 43 44

## 6. Imbalance ablations (one change at a time, five-grade head, natural prevalence)

In [ ]:
import glob
for f in sorted(glob.glob("configs/imbalance/*.yaml")):
    !python -m early_dr.run_experiment $f --set $PATHS --stage train

**Choose on validation only:** best validation score of every run so far (higher is better).
Pick the best sampler, loss and augmentation from this table, then put them in `configs/imbalance/combined.yaml`.

In [ ]:
import glob, json, os
rows = []
for f in sorted(glob.glob('/kaggle/working/runs/*/seed*/train_info.json')):
    rows.append((f.split('/')[-3], f.split('/')[-2], round(json.load(open(f))['best_val_score'], 4)))
for r in rows:
    print(r)

## 7. Extensions (optional, if time allows)
ViT with retinal weights needs the RETFound checkpoint as a Kaggle dataset (edit `pretrained_path`).
For the 768 px ablation first run `prepare_data` with `data.prep_size=768`.

In [ ]:
for f in ["configs/extensions/colour_graham.yaml", "configs/extensions/three_class.yaml"]:
    !python -m early_dr.run_experiment $f --set $PATHS --stage train

In [ ]:
# stage 2: early vs late, DR-positive images only
!python -m early_dr.run_experiment configs/tasks/stage2.yaml --set $PATHS --stage train

## 8. Evaluate on the INTERNAL test sets (DDR test, held-out EyePACS patients)
This is where the validation-selected shortlist is evaluated. Per-image predictions are saved next to the results.

In [ ]:
for name in ["arm_A", "arm_B", "arm_C"]:
    !python -m early_dr.run_experiment configs/arms/{name}.yaml --set $PATHS task=stage1 --stage evaluate --seeds 42 43 44

## 9. Compare arms with Holm-corrected, patient-level bootstrap tests

In [ ]:
!python -m early_dr.summarise --work /kaggle/working --baseline arm_A --experiments arm_B arm_C --source DDR --metric sensitivity
!python -m early_dr.summarise --work /kaggle/working --baseline arm_A --experiments arm_B arm_C --source EyePACS --metric pr_auc

## 10. External test sets LAST (IDRiD, APTOS)
Only after model selection is finished. If **C beats B on its own test split but not on IDRiD and APTOS**, the model learned the source.

In [ ]:
for name in ["arm_A", "arm_B", "arm_C"]:
    !python -m early_dr.run_experiment configs/arms/{name}.yaml --set $PATHS task=stage1 --stage evaluate --external --seeds 42 43 44
!python -m early_dr.summarise --work /kaggle/working --baseline arm_A --experiments arm_B arm_C --source IDRiD --metric auroc
!python -m early_dr.summarise --work /kaggle/working --baseline arm_A --experiments arm_B arm_C --source APTOS --metric auroc

## 11. Explainability: Grad-CAM with the randomised-weights sanity check

In [ ]:
!python -m early_dr.saliency configs/arms/arm_A.yaml --n 20 --set $PATHS task=stage1

## 12. Read the results
Per run: `runs/<experiment>/seed<N>/results.json` (metrics with 95% intervals, per source), `preds_<source>.csv`
(per-image predictions), `calibration.json`, `train_info.json`.

In [ ]:
import json
r = json.load(open("/kaggle/working/runs/arm_A/seed42/results.json"))
for source, res in r.items():
    e = res["early_vs_none"]
    print(f"{source:8s} sensitivity {e['sensitivity']:.3f} {tuple(round(x,3) for x in e['sensitivity_ci'])}  "
          f"specificity {e['specificity']:.3f}  PR-AUC {e['pr_auc']:.3f}  AUROC {e['auroc']:.3f}  "
          f"ECE {res['calibration']['ece']:.3f}")